# RAG LLM Backend Server — Mistral 7B

This notebook runs **Mistral-7B-Instruct-v0.3** behind a FastAPI server and exposes it via localtunnel.

⚠️ Make sure your Colab Runtime is set to **T4 GPU** before running.

### WHY THIS NOTEBOOK EXISTS:
- Your RAG app runs locally on your PC (retriever.py + app.py)
- But the LLM (Mistral 7B) is too big for a CPU — it needs a GPU
- This notebook gives you a FREE T4 GPU from Google Colab
- It starts a server, and your local app sends the prompt here via HTTP

### HOW THE DATA FLOWS:
```
User types question
       ↓
app.py (local PC) retrieves chunks from ChromaDB
       ↓
build_prompt() creates the full prompt
       ↓
HTTP POST → this Colab server
       ↓
Mistral 7B generates the answer
       ↓
answer sent back to app.py → displayed to user
```

In [ ]:
# ============================================================
# CELL 1: Install packages
# ============================================================
# fastapi + uvicorn → web server to receive HTTP requests
# nest-asyncio → allows uvicorn to run inside a Jupyter notebook
# transformers → loads Mistral model and tokenizer from HuggingFace
# accelerate → helps load large models efficiently across GPU memory
# bitsandbytes → enables 4-bit quantization (shrinks 14GB model → ~4GB, fits on T4)
# localtunnel → creates a public URL pointing to our local server port 8000

!pip install -q fastapi uvicorn nest-asyncio transformers accelerate bitsandbytes
!npm install -q -g localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.6 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
added 22 packages in 3s
⠸
⠸3 packages are looking for funding
⠸  run `npm fund` for details
⠸

In [ ]:
# ============================================================
# CELL 2: Load Mistral 7B in 4-bit quantization
# ============================================================
#
# WHY 4-BIT?
#   Mistral-7B at full precision = ~14GB VRAM
#   Google Colab T4 only has 15GB VRAM
#   4-bit quantization compresses it to ~4GB — plenty of room
#   Quality loss is minimal for Q&A tasks
#
# WHY THIS SPECIFIC MODEL (Mistral-7B-Instruct-v0.3)?
#   - "Instruct" means it was fine-tuned to follow instructions
#   - v0.3 supports the apply_chat_template() method correctly
#   - Free to use, no license issues

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "mistralai/Mistral-7B-Instruct-v0.3"

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",             # NF4 = best quality for 4-bit
    bnb_4bit_compute_dtype=torch.float16,  # use fp16 for matrix math
    bnb_4bit_use_double_quant=True,        # extra compression, tiny quality tradeoff
)

print(f"Loading tokenizer for {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading {model_id} in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"  # automatically puts layers on GPU/CPU as needed
)
model.eval()
print("✅ Model loaded successfully!")

Loading tokenizer for mistralai/Mistral-7B-Instruct-v0.3...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

✅ Model loaded successfully!


In [ ]:
# ============================================================
# CELL 3: The generation function — THE MOST IMPORTANT PART
# ============================================================
#
# ⚠️ THE KEY DIFFERENCE: apply_chat_template()
#
# Each model has a "chat template" — a specific format for wrapping
# messages before they become tokens. If you send a raw string like:
#     "SYSTEM: ... USER: ... ASSISTANT:"
# ...the model sees it as plain text and gives bad answers.
#
# Mistral's CORRECT format (what apply_chat_template produces):
#     <s>[INST] system message + user question [/INST]
#
# Qwen's CORRECT format (completely different!):
#     <|im_start|>system\n...\n<|im_end|>\n<|im_start|>user\n...\n<|im_end|>\n<|im_start|>assistant\n
#
# apply_chat_template() handles this AUTOMATICALLY — the tokenizer
# already knows the correct format for its own model.
# This is why retriever.py and app.py can stay the same:
# they send a structured dict {system, context, question},
# and THIS function does the model-specific formatting.

def generate_answer(system_instruction: str, context: str, question: str) -> str:
    """
    Takes the 3 logical parts of a RAG query and generates an answer.

    WHY 3 SEPARATE ARGUMENTS instead of one big prompt string?
    Because apply_chat_template needs them as separate 'roles'.
    The model was trained with role-based messages, not raw strings.

    Parameters:
        system_instruction: What kind of assistant the model should be
        context: The retrieved handbook chunks
        question: The user's actual question
    """
    # Build messages in the standard OpenAI-style role format
    # apply_chat_template will convert this to Mistral's [INST] format
    messages = [
        {
            "role": "system",
            "content": system_instruction
        },
        {
            "role": "user",
            "content": f"HANDBOOK CONTEXT:\n{context}\n\nQUESTION: {question}"
        }
    ]

    # apply_chat_template converts the messages list into the
    # exact token sequence the model expects
    # add_generation_prompt=True appends the [/INST] token that
    # tells the model "now you should generate a response"
    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,           # return as string first, not token IDs
        add_generation_prompt=True
    )

    # Tokenize — now convert the formatted string to token IDs
    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"       # PyTorch tensors for the GPU
    ).to("cuda")

    # Generate the response
    with torch.no_grad():  # no_grad = don't compute gradients (saves memory during inference)
        output_ids = model.generate(
            **inputs,
            max_new_tokens=400,    # max tokens to generate (not input tokens)
            temperature=0.1,       # close to 0 = deterministic, factual answers
            do_sample=True,        # required when temperature != 1.0
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1  # discourages repeating the same sentence
        )

    # Decode ONLY the newly generated tokens (skip the input prompt)
    input_length = inputs["input_ids"].shape[1]
    new_tokens = output_ids[0][input_length:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    return answer

print("✅ Generation function defined.")

✅ Generation function defined.


In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn, nest_asyncio, threading, subprocess, urllib.request, time

server_app = FastAPI()

class GenerateRequest(BaseModel):
    system: str = "You are a helpful academic advisor. Answer only from the context."
    context: str
    question: str

@server_app.post("/generate")
def generate_endpoint(req: GenerateRequest):
    return {"answer": generate_answer(req.system, req.context, req.question)}

@server_app.get("/")
def health():
    return {"status": "running"}

# ── اطبع الـ IP ─────────────────────────────────────────────
public_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode().strip()
print(f"🌐 IP (localtunnel password if asked): {public_ip}")

# ── شغّل uvicorn في thread منفصل (الحل لـ event loop error) ──
def run_server():
    uvicorn.run(server_app, host="0.0.0.0", port=8000, log_level="warning")

threading.Thread(target=run_server, daemon=True).start()
print("✅ Server started on port 8000")
time.sleep(2)  # استنى uvicorn يجهز

# ── شغّل localtunnel وأطبع كل output ──────────────────────
print("\n--- localtunnel output ---")
process = subprocess.Popen(
    ["lt", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)
for line in process.stdout:
    line = line.strip()
    if line:
        print(line)
        if "loca.lt" in line:
            print(f"\n🚀 YOUR URL  →  {line}")
            print(f"   Password  →  {public_ip}")


🌐 IP (localtunnel password if asked): 34.82.96.212
✅ Server started on port 8000

--- localtunnel output ---
your url is: https://nice-games-judge.loca.lt

🚀 YOUR URL  →  your url is: https://nice-games-judge.loca.lt
   Password  →  34.82.96.212
